# Task 2 — Stock Price Prediction (AAPL)

## Objective
Build a regression model to predict **next day Close price** for Apple (AAPL) using engineered features from the last 2 years of daily data.

## Dataset
- Source: `yfinance` (AAPL, last 2 years)
- Frequency: daily trading bars

## Approach
1. Download data via `yfinance`
2. Engineer features: Open/High/Low/Volume, lagged Close (t-1, t-2), rolling means (5-day, 20-day)
3. Target: next day's Close (t+1)
4. Time-series split (first 80% train, last 20% test; no shuffle)
5. Train Random Forest Regressor (fallback: Linear Regression)
6. Evaluate with MAE, RMSE, R² and visualize predictions and residuals

## Final Summary
This notebook trains a baseline ML model for next-day close prediction and reports standard regression metrics and plots.

In [ ]:
# pip install yfinance pandas numpy scikit-learn matplotlib seaborn

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
def download_aapl_last_2y():
    try:
        df = yf.download('AAPL', period='2y', interval='1d', auto_adjust=False, progress=False)
    except Exception as e:
        raise RuntimeError(f'Failed to download data from yfinance: {e}')

    if df is None or df.empty:
        raise RuntimeError('No data returned from yfinance. Check network access or ticker symbol.')

    df = df.dropna().copy()
    df.index = pd.to_datetime(df.index)
    return df

df = download_aapl_last_2y()
df.head()

In [ ]:
def make_features_and_target(raw_df: pd.DataFrame) -> pd.DataFrame:
    data = raw_df.copy()

    # Lagged close
    data['Close_lag1'] = data['Close'].shift(1)
    data['Close_lag2'] = data['Close'].shift(2)

    # Rolling means of close
    data['Close_roll5'] = data['Close'].rolling(window=5).mean()
    data['Close_roll20'] = data['Close'].rolling(window=20).mean()

    # Target: next day's close
    data['Target_Close_t+1'] = data['Close'].shift(-1)

    # Keep required feature columns
    keep_cols = [
        'Open', 'High', 'Low', 'Volume',
        'Close_lag1', 'Close_lag2',
        'Close_roll5', 'Close_roll20',
        'Target_Close_t+1'
    ]

    data = data[keep_cols].dropna().copy()
    return data

data = make_features_and_target(df)
data.head()

In [ ]:
feature_cols = ['Open', 'High', 'Low', 'Volume', 'Close_lag1', 'Close_lag2', 'Close_roll5', 'Close_roll20']
target_col = 'Target_Close_t+1'

X = data[feature_cols].values
y = data[target_col].values

split_idx = int(len(data) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

len(X_train), len(X_test)

In [ ]:
def train_model(X_train, y_train):
    # Prefer RandomForest; fallback to LinearRegression if something goes wrong
    try:
        model = RandomForestRegressor(
            n_estimators=400,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        return model, 'RandomForestRegressor'
    except Exception as e:
        print(f'RandomForest failed ({e}). Falling back to LinearRegression.')
        model = LinearRegression()
        model.fit(X_train, y_train)
        return model, 'LinearRegression'

model, model_name = train_model(X_train, y_train)
model_name

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f'Model: {model_name}')
print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2:  {r2:.4f}')

In [ ]:
# Plot: Actual vs Predicted
plt.figure(figsize=(12, 5))
plt.plot(y_test, label='Actual', linewidth=2)
plt.plot(y_pred, label='Predicted', linewidth=2)
plt.title('AAPL Next-Day Close: Actual vs Predicted')
plt.xlabel('Test Time Index')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plot: Residuals
residuals = y_test - y_pred
plt.figure(figsize=(10, 5))
sns.histplot(residuals, bins=30, kde=True)
plt.title('Residual Distribution (Actual - Predicted)')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()


## Notes
- This is a simple baseline and does **not** guarantee profitable trading.
- Time series forecasting is sensitive to regime changes; consider adding more features (returns, technical indicators) and stronger validation (walk-forward).